In [ ]:
%tensorflow_version 1.x
import tensorflow as tf
print(tf.__version__)

In [ ]:
# Original Google Drive mount removed; supply local data instead.


In [ ]:
import os
os.chdir('..')

In [ ]:
% cd '.'


* Install the GCN libraries by Defferard.
* These were used by Jia et al. as a template to write their code.


In [ ]:
from Lib import GCN_Model, graph, coarsening

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import sparse
device = 'cuda:0'
%matplotlib inline

In [ ]:
# Clear all the stack and use GPU resources as much as possible
tf.reset_default_graph()
config = tf.ConfigProto()
config.gpu_options.allow_growth = True
sess = tf.Session(config=config)

In [ ]:
train_data = pd.read_csv ('data/Region1/training_set.csv', header=None)
train_labels = pd.read_csv ('data/Region1/training_label.csv', header=None)
val_data = pd.read_csv ('data/Region1/val_set.csv', header=None)
val_labels = pd.read_csv ('data/Region1/val_label.csv', header=None)
test_data = pd.read_csv ('data/Region1/test_set.csv', header=None)
test_labels = pd.read_csv ('data/Region1/test_label.csv', header=None)

In [ ]:
train_data = train_data.to_numpy()
train_data = train_data.astype('float32')
train_labels = train_labels.to_numpy()
train_labels = train_labels.astype('int32')
val_data = val_data.to_numpy()
val_data = val_data.astype('float32')
val_labels = val_labels.to_numpy()
val_labels = val_labels.astype('int32')
test_data = test_data.to_numpy()
test_data = test_data.astype('float32')
test_labels = test_labels.to_numpy()
test_labels = test_labels.astype('int32')
train_labels=train_labels.reshape(430080,)
val_labels= val_labels.reshape(53760,)
test_labels= test_labels.reshape(53760,)


In [ ]:
print(train_data.shape)
print(train_labels.shape)
print(val_data.shape)
print(val_labels.shape)

In [ ]:
type(train_data)

In [ ]:
Adjacency_Matrix = pd.read_csv ('data/All Electrodes/Adjacency_Matrix.csv', header=None,float_precision=32)
Adjacency_Matrix = Adjacency_Matrix.to_numpy()
Adjacency_Matrix = Adjacency_Matrix.astype('float32')
Adjacency_Matrix = sparse.csr_matrix(Adjacency_Matrix)

print('==============> Adjancy matrix read!')
print(Adjacency_Matrix.shape)

In [ ]:
graphs, perm = coarsening.coarsen(Adjacency_Matrix, levels=5, self_connections=False)


In [ ]:
L = [graph.laplacian(A, normalized=True) for A in graphs]
graph.plot_spectrum(L)

In [ ]:
X_train = coarsening.perm_data(train_data, perm)
X_val  = coarsening.perm_data(val_data,  perm)
X_test =  coarsening.perm_data(test_data,  perm)
print('==============>coarsening done!')

In [ ]:
print(train_labels[1:20])

In [ ]:

params = dict()
params['dir_name']       = 'demo'
params['num_epochs']     = 50
params['batch_size']     = 1024
params['eval_frequency'] = 100

# Building blocks.
params['filter'] = 'chebyshev5'
params['brelu']  = 'b2relu'
params['pool']   = 'mpool1'

# Architecture.
params['F']              = [16,32,64,128,256,512]  # Number of graph convolutional filters.
params['K']              = [2,2,2,2,2,2]  # Polynomial orders.
params['p']              = [2,2,2,2,2,2]    # Pooling sizes.
params['M']              = [4]  # Output dimensionality of fully connected layers.

# Optimization.
params['regularization'] = 0.001     # L2 regularization
params['dropout']        = 0.50      # Dropout rate
params['learning_rate']  = 0.0001  # Learning rate
params['decay_rate']     = 1         # Learning rate Decay == 1 means no Decay
params['momentum']       = 0         # momentum == 0 means Use Adam Optimizer
n_train = np.shape(train_data)[0]
params['decay_steps']    = n_train/ params['batch_size']

In [ ]:
!pwd

In [ ]:
model = GCN_Model.cgcnn(L, **params)


In [ ]:
accuracy, loss, t_step = model.fit(X_train, train_labels, X_val, val_labels)

In [ ]:
results = model.evaluate(X_test, test_labels)


In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 5))
ax1.plot(accuracy, 'b.-')
ax1.set_ylabel('validation accuracy', color='b')
ax1.set_ylim([0,100])
ax2 = ax1.twinx()
ax2.plot(loss, 'g.-')
ax2.set_ylabel('training loss', color='g')
plt.show()


In [ ]:
import pandas
df = pandas.DataFrame(data={"Results": results})
df.to_csv("./Results_Imag_NoFilterBR1.csv", sep=',',index=False)
df = pandas.DataFrame(data={"Loss": loss, "Accuracy": accuracy})
df.to_csv("./LA_Imag_NoFilterBR1.csv", sep=',',index=False)